# SHAP: a receipt for every prediction

A bank's model looks at Maya's loan application and prints one number. Say, **"76% likely to repay."**

Maya asks the obvious question: **why did it say that?**

The model does not answer. It is like a restaurant bill that shows only the total, with no list of what you ordered.

**SHAP is the itemized receipt.** It shows where the model started, and how far each fact about Maya pushed the number up or down.

*Companion page:* [The SHAP Receipt](https://isom-835.vercel.app/explore/shap/) tells the same story with sliders you can drag. Try it before or after this notebook.

---

### How to use this notebook

1. Click **Runtime → Run all** in the menu. Or press the ▶ play button on each gray code cell, from top to bottom.
2. Read the text, run the cell, look at what it prints. That is the whole method.
3. You do not need to know how to code. Every code cell has a plain-English note above it.

### The road map

| Part | What happens | Big idea |
|---|---|---|
| 1 | A trivia night | How to split credit fairly: the **Shapley value** |
| 2 | We build a loan model | A **black box** that gives a number and no reasons |
| 3 | Maya's receipt, by hand | SHAP is the trivia split, applied to a prediction |
| 4 | Maya's receipt, in one line | The `shap` library |
| 5 | Checking the receipt | The fairness rules |
| 6 | A stack of receipts | How the whole model thinks |
| 7 | Sliders | Play with it |
| 8 | The fine print | What SHAP does **not** tell you |
| 9 | Your turn | Three small challenges |

## Part 0. Setup

This cell installs the `shap` library and loads the tools we need. It takes about 20 seconds. When it prints **Ready**, move on.

In [ ]:
!pip -q install shap

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

print("Ready. shap version:", shap.__version__)

---
## Part 1. A trivia night

SHAP is built on an older idea from game theory. In 1953, Lloyd Shapley asked:

> A team wins a prize together. How much did each player earn?

Meet the team:

| Player | Knows | Score alone |
|---|---|---|
| **Ali** | history and science | 40 |
| **Bea** | history and sports | 50 |
| **Cy** | movies and music | 10 |

The cell below is the scoreboard. It lists the score for **every possible team**, from the empty table (0 points) to all three together (90 points).

In [ ]:
score = {
    ():                     0,    # empty table
    ("Ali",):              40,
    ("Bea",):              50,
    ("Cy",):               10,
    ("Ali", "Bea"):        70,    # not 90: they both know history, so their knowledge overlaps
    ("Ali", "Cy"):         60,    # more than 40 + 10: Cy covers what Ali cannot
    ("Bea", "Cy"):         60,
    ("Ali", "Bea", "Cy"):  90,    # the full team
}

def team_score(team):
    return score[tuple(sorted(team))]

print("Ali alone:        ", team_score(["Ali"]))
print("Ali + Bea:        ", team_score(["Ali", "Bea"]))
print("Ali + Bea + Cy:   ", team_score(["Ali", "Bea", "Cy"]))

### The catch: credit depends on who sat down first

Seat the players **one at a time**. Pay each player whatever the score went up when they arrived. That raise has a name: the **marginal contribution**.

The next cell does this for one seating order. **Try it:** change the order inside the square brackets and run the cell again.

In [ ]:
def paychecks(order):
    # Seat players one by one. Each gets the amount the score went up when they arrived.
    table = []
    pay = {}
    for player in order:
        before = team_score(table)
        table.append(player)
        after = team_score(table)
        pay[player] = after - before
    return pay

order = ["Ali", "Bea", "Cy"]      # <-- change the order and run again

for player, amount in paychecks(order).items():
    print(f"{player} arrives and adds {amount}")

Same team, same 90 points, **different paychecks** every time you change the order. No single order is fair.

### Shapley's move: use every order, then average

Three players can sit down in 6 different orders. The next cell tries all 6 and averages each player's paycheck.

In [ ]:
players = ["Ali", "Bea", "Cy"]

rows = []
for order in itertools.permutations(players):
    pay = paychecks(order)
    rows.append({"seating order": " → ".join(order), **pay})

table = pd.DataFrame(rows).set_index("seating order")[players]
table.loc["AVERAGE = Shapley value"] = table.mean()
table.round(1)

### What to notice

- The bottom row is each player's **Shapley value**: Ali 35, Bea 40, Cy 15.
- They add up to exactly **90**. Nothing is lost, nothing is invented.
- **Cy** scores only 10 alone, but earns 15. He knows things nobody else knows.
- **Ali** scores 40 alone, but earns 35. Some of what he knows, Bea knows too.

A fair split pays you for **what you add to the team**. Keep that in mind. It is the whole idea.

---
## Part 2. Build a black box

Now we leave the trivia night and build a loan model.

First we need applicants. The cell below invents 6,000 of them. Each one has five facts (in machine learning, facts are called **features**):

| Feature | Meaning |
|---|---|
| `income` | yearly income, in thousands of dollars |
| `credit_years` | years since their first credit account |
| `missed_payments` | payments missed in the last two years |
| `job_years` | years at their current job |
| `shoe_size` | their shoe size. Yes, really. It has **nothing** to do with repaying. Remember it for later. |

The last column, `repaid`, says whether the person paid the loan back (1) or not (0).

In [ ]:
rng = np.random.default_rng(7)
n = 6000

people = pd.DataFrame({
    "income":          np.clip(np.exp(rng.normal(4.05, 0.45, n)), 20, 160).round(),
    "credit_years":    np.clip(rng.normal(8, 5, n), 0, 30).round(),
    "missed_payments": rng.choice([0, 1, 2, 3, 4], size=n, p=[0.55, 0.25, 0.12, 0.05, 0.03]),
    "job_years":       np.clip(rng.exponential(4.5, n), 0, 20).round(),
    "shoe_size":       np.clip(rng.normal(9, 1.5, n), 5, 13).round(),
}).astype(int)

# How the (invented) world decides who repays. Shoe size plays no part.
z = (-0.045
     + 0.028 * (people["income"] - 60)
     + 0.10  * (people["credit_years"] - 8)
     - 0.55  * people["missed_payments"]
     - 0.012 * people["missed_payments"] * np.maximum(0, 75 - people["income"])
     + 0.08  * (people["job_years"] - 4))
true_chance = 1 / (1 + np.exp(-z))
repaid = (rng.random(n) < true_chance).astype(int)      # the world also has luck in it

features = list(people.columns)
people.assign(repaid=repaid).head()

### Train the model

The next cell trains a **gradient-boosted tree model**. Think of it as 150 small flowcharts that vote. It learns from 4,500 applicants and is tested on 1,500 it has never seen.

Nobody can read 150 flowcharts by eye. That is what people mean by a **black box**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(people, repaid, test_size=0.25, random_state=0)

model = GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=0.05,
                                   subsample=0.8, min_samples_leaf=40, random_state=0)
model.fit(X_train, y_train)

def chance_of_repaying(applicants):
    # Ask the black box. Returns a number from 0 to 100 for each applicant.
    applicants = pd.DataFrame(applicants, columns=features)
    return model.predict_proba(applicants)[:, 1] * 100

best_possible = ((true_chance[X_test.index] > 0.5) == y_test).mean()
print(f"The model is right about {model.score(X_test, y_test):.0%} of applicants it has never seen.")
print(f"The world we invented has luck in it. Even knowing the hidden rule gets you only about {best_possible:.0%}.")
print("So the model is doing about as well as anyone could.")

### Meet Maya

Here is Maya's application. We hand it to the black box.

In [ ]:
maya = {"income": 95, "credit_years": 12, "missed_payments": 1, "job_years": 6, "shoe_size": 8}

maya_score = chance_of_repaying([maya])[0]
print(f"The model says: Maya is {maya_score:.1f}% likely to repay.")
print("Reasons given: none.")

---
## Part 3. Maya's receipt, by hand

Now the key step. We play **the trivia game again**, with a new cast:

| Trivia night | Loan model |
|---|---|
| The players | The **features**: income, credit history, and so on |
| The team's score | The **prediction** |
| The empty table | The **base value**: the model's average prediction for a crowd of applicants, before it knows anything about Maya |
| A player sits down | The model gets to see **Maya's real value** for that feature |
| An empty chair | The model fills that feature in with **other people's values** and averages |
| A player's fair share | That feature's **SHAP value** |

### Step 1. Pick the crowd

The "empty chair" needs other people's values. So we pick 100 applicants at random. This comparison group is called the **background data**.

In [ ]:
crowd = X_train.sample(100, random_state=1)

base_value = chance_of_repaying(crowd).mean()
print(f"Base value: {base_value:.1f}%")
print("This is the model's average answer for the crowd. It is where every receipt starts.")

### Step 2. Score any "team" of features

This is the scoreboard for the new game.

`score_with(maya, team)` means: let the model see Maya's real values **only for the features on the team**. For every other feature, use the crowd's values. Then average the model's answers.

In [ ]:
def score_with(person, team):
    mixed = crowd.copy()
    for feature in team:
        mixed[feature] = person[feature]      # this feature "sits down": use the person's real value
    return chance_of_repaying(mixed).mean()

print(f"Empty table (knows nothing about Maya):   {score_with(maya, []):.1f}%")
print(f"Knows only Maya's income:                 {score_with(maya, ['income']):.1f}%")
print(f"Knows income and missed payments:         {score_with(maya, ['income', 'missed_payments']):.1f}%")
print(f"Full team (knows everything):             {score_with(maya, features):.1f}%")

### What to notice

- The empty table gives the **base value**.
- The full team gives **Maya's prediction**.
- Everything in between is a partial team, like Ali and Bea without Cy.

### Step 3. One arrival order

Features arrive one at a time. Each one gets credit for how far it moved the score. **Try it:** shuffle the order and run again. The steps change size, like the paychecks did.

In [ ]:
def steps(person, order):
    table = []
    credit = {}
    for feature in order:
        before = score_with(person, table)
        table.append(feature)
        after = score_with(person, table)
        credit[feature] = after - before
    return credit

order = ["income", "credit_years", "missed_payments", "job_years", "shoe_size"]   # <-- shuffle me

print(f"Start at the base value: {base_value:.1f}")
for feature, move in steps(maya, order).items():
    print(f"  {feature:<16} arrives and moves the score by {move:+.1f}")
print(f"End at Maya's prediction: {maya_score:.1f}")

### Step 4. Every order, then average

Five features can arrive in **120** different orders. We try all of them and average. This is exactly what we did for Ali, Bea and Cy.

The averages are Maya's **SHAP values**. (This cell takes a few seconds.)

In [ ]:
def shap_by_hand(person):
    total = {feature: 0.0 for feature in features}
    all_orders = list(itertools.permutations(features))
    for order in all_orders:
        for feature, move in steps(person, order).items():
            total[feature] += move
    return {feature: total[feature] / len(all_orders) for feature in features}

def print_receipt(person, base, values, title="SHAP RECEIPT"):
    line = "-" * 46
    print(f"{title:^46}")
    print(line)
    print(f"{'Base value (average applicant)':<36}{base:>9.1f}%")
    print(line)
    for feature, value in values.items():
        label = f"{feature} = {person[feature]}"
        print(f"{label:<36}{value:>+10.1f}")
    print(line)
    print(f"{'Prediction':<36}{base + sum(values.values()):>9.1f}%")

maya_values = shap_by_hand(maya)
print_receipt(maya, base_value, maya_values)

### How to read the receipt

- Start at the **base value**. That is the model's answer before it knows anything about Maya.
- Each line moves the score. A **plus** pushes it up. A **minus** pushes it down.
- The last line is the model's answer for Maya. The lines add up to it.

That is SHAP. You just computed it yourself, with nothing but a loop and an average.

---
## Part 4. Maya's receipt, in one line

Nobody does this by hand in real life. The `shap` library does it for you.

You give it two things: **the model** and **the crowd**. Then you hand it a person.

In [ ]:
explainer = shap.Explainer(chance_of_repaying, crowd)        # the model + the crowd
maya_explained = explainer(pd.DataFrame([maya]))             # explain Maya

compare = pd.DataFrame({
    "by hand":      maya_values,
    "shap library": dict(zip(features, maya_explained.values[0])),
})
compare.round(2)

The two columns match. The library is doing what you did, only faster.

### The waterfall plot

This is the receipt as a picture. Read it **from the bottom up**:

- The gray number at the bottom, `E[f(X)]`, is the **base value**.
- <span style="color:#ff0051">**Red**</span> bars push the score **up**. <span style="color:#008bfb">**Blue**</span> bars push it **down**.
- The number at the top, `f(x)`, is **Maya's prediction**.

In [ ]:
shap.plots.waterfall(maya_explained[0])

---
## Part 5. Checking the receipt

Why trust this split and not some other one? Because Shapley's average is the only split that keeps a set of common-sense fairness rules all at once. Two of them we can check right here.

### Rule 1. It adds up

Base value + all the SHAP values = the prediction. Always.

In [ ]:
total_from_receipt = maya_explained.base_values[0] + maya_explained.values[0].sum()

print(f"Base value + SHAP values = {total_from_receipt:.4f}")
print(f"Model's prediction       = {maya_score:.4f}")

### Rule 2. No free riders

A feature that never changes the prediction gets **exactly zero**.

We know shoe size has nothing to do with repaying. So what does Maya's receipt say about it?

In [ ]:
print(f"Maya's shoe size SHAP value: {maya_values['shoe_size']:+.2f} points")
print()
print("Biggest shoe-size effect the model has for anyone in the test group:")
shoe_effect = explainer(X_test.head(200)).values[:, features.index("shoe_size")]
print(f"  {np.abs(shoe_effect).max():.2f} points")

### What to notice

The number is small, but it is probably **not exactly zero**. Why?

Because SHAP explains **the model**, and this model is not perfect. While training, it found a few accidental patterns in shoe size and kept them. SHAP caught that and reported it.

This is one of SHAP's best uses: **debugging**. If a feature that should not matter shows up on the receipt, your model has learned something it should not have.

Two more rules, in words:

- **Equal work, equal credit.** If two features always help by the same amount, they get the same share.
- **More work, never less credit.** If you change a model so it leans on a feature more, that feature's SHAP value never goes down.

---
## Part 6. A stack of receipts

One receipt explains **one decision**. Now we print a receipt for 300 applicants and stack them. This shows how the model thinks in general.

In [ ]:
group = X_test.head(300)
everyone_explained = explainer(group)
print("Receipts printed:", len(group))

### The beeswarm plot

Each dot is **one applicant**. Each row is one feature.

- Dots on the **right**: the feature pushed that person's score **up**.
- Dots on the **left**: it pushed their score **down**.
- **Color** shows the person's own value: <span style="color:#ff0051">red = high</span>, <span style="color:#008bfb">blue = low</span>.

In [ ]:
shap.plots.beeswarm(everyone_explained)

### What to notice

- **income**: red dots (high income) are on the right. The model rewards income.
- **missed_payments**: red dots (many missed payments) are far to the left. The model punishes them.
- **shoe_size**: every dot is squeezed against zero. The model barely uses it.

### The bar plot: which features matter most?

This takes the average size of each feature's push, ignoring direction. It is the standard way to measure **feature importance** with SHAP.

In [ ]:
shap.plots.bar(everyone_explained)

### The scatter plot: one feature up close

Each dot is one applicant. Left to right is their income. Up and down is how much income moved their score. The color shows their missed payments.

Read it like this: as income goes up, the dots climb. More income, bigger push up.

In [ ]:
shap.plots.scatter(everyone_explained[:, "income"], color=everyone_explained[:, "missed_payments"])

---
## Part 7. Play with it

Move the sliders. The model re-scores the applicant and SHAP prints a new receipt.

**Things to try**

- Slide `shoe_size` from end to end. Watch how little the receipt cares.
- Push `missed_payments` up to 4. Which line takes the hit?
- Find the lowest score you can. Then read the receipt to see why.

(If the sliders do not appear, run this cell again.)

In [ ]:
from ipywidgets import interact, IntSlider

def live_receipt(income=95, credit_years=12, missed_payments=1, job_years=6, shoe_size=8):
    person = {"income": income, "credit_years": credit_years, "missed_payments": missed_payments,
              "job_years": job_years, "shoe_size": shoe_size}
    explained = explainer(pd.DataFrame([person]))
    values = dict(zip(features, explained.values[0]))
    print_receipt(person, explained.base_values[0], values)

interact(live_receipt,
         income=IntSlider(value=95, min=20, max=160, description="income"),
         credit_years=IntSlider(value=12, min=0, max=30, description="credit_years"),
         missed_payments=IntSlider(value=1, min=0, max=4, description="missed_pay"),
         job_years=IntSlider(value=6, min=0, max=20, description="job_years"),
         shoe_size=IntSlider(value=8, min=5, max=13, description="shoe_size"));

---
## Part 8. The fine print

A receipt is honest about what you were charged. It says nothing about whether the prices were right. Three limits to remember.

### 1. SHAP explains the model, not the world

"Income added 25 points" means **the model leans on Maya's income**. It does not mean that giving Maya a raise would make her repay. That is a cause-and-effect question, and SHAP does not answer it.

### 2. Features that overlap share the credit

If two features carry the same information (say, income and home value), the split between them can look odd. It is Ali and Bea's shared history knowledge again.

### 3. The receipt depends on the crowd

The base value is "the average applicant". Change who is in the crowd and the receipt changes. The next cell proves it: we explain Maya again, but compare her only with **high-income** applicants.

In [ ]:
rich_crowd = X_train[X_train["income"] >= 90].sample(100, random_state=1)
rich_explainer = shap.Explainer(chance_of_repaying, rich_crowd)
maya_vs_rich = rich_explainer(pd.DataFrame([maya]))

print_receipt(maya, base_value, maya_values, title="MAYA vs. EVERYONE")
print()
print_receipt(maya, maya_vs_rich.base_values[0], dict(zip(features, maya_vs_rich.values[0])),
              title="MAYA vs. HIGH EARNERS")

### What to notice

Same Maya, same model, same prediction at the bottom. But among high earners, Maya's income is nothing special. Its line is no longer a big plus, and it can even turn into a minus. The base value is higher too, because high earners score higher on average.

A SHAP value always answers the question: **compared with whom?**

### A note on speed

We tried every order of 5 features: 120 orders. With 30 features there are more orders than stars in the observable universe. Real tools take shortcuts:

- **KernelSHAP** tries a sample of teams and estimates.
- **TreeSHAP** uses the structure of tree models to get exact answers quickly.

The `shap` library picks a sensible method for you.

---
## Part 9. Your turn

### Challenge 1. A new applicant

Change the numbers below to describe someone else. Run the cell and read their receipt.

In [ ]:
someone = {"income": 45, "credit_years": 3, "missed_payments": 2, "job_years": 1, "shoe_size": 10}   # <-- edit me

explained = explainer(pd.DataFrame([someone]))
print_receipt(someone, explained.base_values[0], dict(zip(features, explained.values[0])))

### Challenge 2. The lowest score in the group

The cell below finds the applicant with the lowest score among the 300 and draws their waterfall. **Before you run it, guess:** which feature will be the biggest blue bar?

In [ ]:
scores = chance_of_repaying(group)
lowest = int(np.argmin(scores))

print(group.iloc[lowest].to_string())
print(f"\nModel's score: {scores[lowest]:.1f}%")
shap.plots.waterfall(everyone_explained[lowest])

### Challenge 3. Three questions to answer in your own words

1. What is the **base value**, and where does it come from?
2. Shoe size got a tiny SHAP value that was not zero. What does that tell you about **the model**?
3. Income has a big positive SHAP value for Maya. Is it safe to say "if Maya earned more, she would be more likely to repay"? Why or why not?

<details>
<summary><b>Click to check your answers</b></summary>

1. It is the model's average prediction for the crowd (the background data). It is the "empty table": the score before the model knows anything about the person.
2. The model picked up a little accidental pattern in shoe size during training. SHAP reports what the model does, flaws included.
3. Not safe. SHAP describes how the model used her income. It makes no claim about cause and effect in the real world.

</details>

---
## The whole idea in one breath

> Treat the features as teammates. Pay each one its average contribution across every order they could arrive in. Hand over a receipt that adds up to the prediction.

### Words you now know

| Term | Plain meaning |
|---|---|
| **Feature** | One fact about a person that the model uses |
| **Black box** | A model that gives answers without reasons |
| **Shapley value** | A player's fair share: their average contribution over every arrival order |
| **Marginal contribution** | How much the score moved when one player (or feature) arrived |
| **Base value** | The model's average prediction for the crowd. Where every receipt starts |
| **Background data** | The crowd you compare a person with |
| **SHAP value** | One line on the receipt: how far a feature pushed this prediction up or down |
| **Feature importance** | The average size of a feature's push across many people |

### Sources

- L. S. Shapley, "A Value for n-Person Games" (1953)
- S. M. Lundberg and S.-I. Lee, "A Unified Approach to Interpreting Model Predictions" (2017)
- The `shap` library: https://shap.readthedocs.io

*The applicants and the loan model in this notebook are invented for teaching.*